# Chapter 3 - Lab 5: <font color='blue'>OpenAI Agents SDK</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using OpenAI Agents SDK** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

The OpenAI Agents SDK is the company's first-party, lightweight agent runtime built directly on top of function calling. Its **API surface is intentionally tiny**: one `Agent`, one `Runner`, and a `@function_tool` decorator.

In this lab you will see how the decorator pulls tool schemas straight from your Python type hints, and how `Runner.run_sync` reduces the agent loop to a single call.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **OpenAI Agents SDK** (`openai-agents`) — `Agent`, `Runner`, `@function_tool`.
* **OpenAI** `gpt-4o-mini` — model used by default.
* **Type hints + docstrings** — these *are* the tool schema.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q openai-agents pydantic python-dotenv

## 2. Set up the API key(s)

This lab needs the following key(s):

  * **`OPENAI_API_KEY`** — your OpenAI key

If you are running this notebook in **Google Colab**, add each key in the left vertical menu under the *key* icon, using the names above.

If you are running locally, set the same names as environment variables (or load them from a `.env` file).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume env vars are already set (e.g. via .env).
    pass

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Declare tools with `@function_tool`

We wrap our two shared functions with `@function_tool`. The SDK reads the function signature and docstring to build the JSON schema sent to the model — no schema file to maintain, no Pydantic glue to write.

In [ ]:
from agents import Agent, Runner, function_tool

@function_tool
def get_stock_data_tool(ticker: str) -> str:
    """Get stock data including price and EPS for a given ticker symbol.

    Args:
        ticker: Stock ticker symbol (e.g., AAPL, JPM)
    Returns:
        String containing stock price and EPS data
    """
    result = get_stock_data(ticker)
    return f'Stock data for {ticker}: Price=${result.price}, EPS=${result.eps}'


@function_tool
def compute_pe_tool(price: float, eps: float) -> str:
    """Compute P/E ratio given stock price and earnings per share.

    Args:
        price: Stock price
        eps:   Earnings per share
    Returns:
        Human-readable P/E ratio.
    """
    return f'P/E ratio: {compute_pe(price, eps)}'

## 5. Build the agent

The `Agent` constructor takes a name, the system-style `instructions`, the tools, and a model name. That is the entire configuration surface for a basic agent.

In [ ]:
agent = Agent(
    name='Financial Analysis Agent',
    instructions=system_message,
    tools=[get_stock_data_tool, compute_pe_tool],
    model='gpt-4o-mini',
)

## 6. Run the agent

`Runner.run_sync` blocks until the agent finishes its loop and returns a result object. For an async variant use `Runner.run`.

In [ ]:
result = Runner.run_sync(agent, input_message)
print(result.final_output)

## 7. Results

You should see the agent call `get_stock_data_tool` (once per ticker), then `compute_pe_tool` (once per ticker), and finally produce a short memo comparing the two.

**What to notice about OpenAI Agents SDK specifically:**

* The smallest API surface of any framework in this chapter — three imports get you a working agent.
* Tool schemas come from your type hints and docstrings, which keeps definitions DRY and forces good documentation.
* Native to OpenAI's API conventions — minimal abstraction overhead, which matters when you are debugging in production.
* Trade-off: ties you to OpenAI's runtime conventions. For multi-provider flexibility, LangChain or AutoGen are easier swap-friendly.